# 01 — Data Profiling

Profil awal `IndonesiaAI/cleaned-data-split-0` memakai Polars LazyFrame dan DuckDB. Sumber, lisensi, dan struktur data dijelaskan pada [`data/README.md`](../data/README.md).

> Dataset tersimpan sebagai enam shard di `data/raw/data/`, bukan sebagai satu `dataset.parquet`.

In [1]:
from pathlib import Path

import duckdb
import polars as pl
import pyarrow.parquet as pq

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'data').exists() else cwd.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'data'
PARQUET_FILES = sorted(RAW_DIR.glob('*.parquet'))
if not PARQUET_FILES:
    raise FileNotFoundError(f'Tidak ada shard Parquet di: {RAW_DIR}')
print(f'Folder data: {RAW_DIR}')
print(f'Jumlah shard: {len(PARQUET_FILES)}')

Folder data: D:\semester 7\tugas1-Luqmanul-Hakim-141-2026\data\raw\data
Jumlah shard: 6


## Validasi file dan ukuran

Validasi metadata dilakukan sebelum query. Shard yang rusak dicatat dan tidak dipakai untuk statistik agar hasil tidak menyesatkan.

In [2]:
valid_files, invalid_files = [], []
for file in PARQUET_FILES:
    try:
        pq.ParquetFile(file).schema_arrow
        valid_files.append(file)
    except Exception as error:
        invalid_files.append({'file': file.name, 'error': str(error).splitlines()[0]})

total_size_mb = sum(file.stat().st_size for file in PARQUET_FILES) / 1_000_000
valid_size_mb = sum(file.stat().st_size for file in valid_files) / 1_000_000
print(f'Ukuran semua shard: {total_size_mb:,.1f} MB')
print(f'Shard valid: {len(valid_files)}; ukuran terbaca: {valid_size_mb:,.1f} MB')
invalid_files
if not valid_files:
    raise RuntimeError('Tidak ada shard Parquet yang valid.')

Ukuran semua shard: 960.0 MB
Shard valid: 5; ukuran terbaca: 762.8 MB


## Profil Polars

`scan_parquet` hanya membangun query plan; file dibaca ketika `collect()` dipanggil.

In [3]:
lazy_data = pl.scan_parquet([str(file) for file in valid_files])
schema = lazy_data.collect_schema()
print('Tipe kolom:')
for name, dtype in schema.items():
    print(f'- {name}: {dtype}')

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

Tipe kolom:
- qid: Int64
- question: String
- date: String
- metadata: List(String)
- response_j: String
- response_k: String


jumlah_baris,qid_null,question_null,date_null,metadata_null,response_j_null,response_k_null
u32,u32,u32,u32,u32,u32,u32
875214,0,0,0,0,0,0


In [4]:
contoh_data = lazy_data.head(10).collect()
contoh_data

qid,question,date,metadata,response_j,response_k
i64,str,str,list[str],str,str
63110,"""I'm curious what John means in…","""2021/07/06""","[""https://hermeneutics.stackexchange.com/questions/63110"", ""https://hermeneutics.stackexchange.com"", ""https://hermeneutics.stackexchange.com/users/43733/""]","""There are a few references to …","""**Is Judas cursed to eternal d…"
63110,"""I'm curious what John means in…","""2021/07/06""","[""https://hermeneutics.stackexchange.com/questions/63110"", ""https://hermeneutics.stackexchange.com"", ""https://hermeneutics.stackexchange.com/users/43733/""]","""There are a few references to …","""No. I can see that God created…"
63110,"""I'm curious what John means in…","""2021/07/06""","[""https://hermeneutics.stackexchange.com/questions/63110"", ""https://hermeneutics.stackexchange.com"", ""https://hermeneutics.stackexchange.com/users/43733/""]","""Here is the meaning of the wor…","""**Is Judas cursed to eternal d…"
63110,"""I'm curious what John means in…","""2021/07/06""","[""https://hermeneutics.stackexchange.com/questions/63110"", ""https://hermeneutics.stackexchange.com"", ""https://hermeneutics.stackexchange.com/users/43733/""]","""Here is the meaning of the wor…","""No. I can see that God created…"
994,"""At the very end of House of Le…","""2017/01/25""","[""https://literature.stackexchange.com/questions/994"", ""https://literature.stackexchange.com"", ""https://literature.stackexchange.com/users/-1/""]","""Yggdrasil is an Ash Tree that …","""I would argue that it's one fi…"
994,"""At the very end of House of Le…","""2017/01/25""","[""https://literature.stackexchange.com/questions/994"", ""https://literature.stackexchange.com"", ""https://literature.stackexchange.com/users/-1/""]","""Yggdrasil is an Ash Tree that …","""The World Ash Tree of Norse my…"
994,"""At the very end of House of Le…","""2017/01/25""","[""https://literature.stackexchange.com/questions/994"", ""https://literature.stackexchange.com"", ""https://literature.stackexchange.com/users/-1/""]","""Yggdrasil is an Ash Tree that …","""There is a chance the referenc…"
994,"""At the very end of House of Le…","""2017/01/25""","[""https://literature.stackexchange.com/questions/994"", ""https://literature.stackexchange.com"", ""https://literature.stackexchange.com/users/-1/""]","""The World Ash Tree of Norse my…","""I would argue that it's one fi…"
994,"""At the very end of House of Le…","""2017/01/25""","[""https://literature.stackexchange.com/questions/994"", ""https://literature.stackexchange.com"", ""https://literature.stackexchange.com/users/-1/""]","""There is a chance the referenc…","""I would argue that it's one fi…"


## Kualitas awal

Tidak ada baris yang diubah atau dihapus pada tahap ini.

In [5]:
quality_profile = lazy_data.with_columns(
    pl.col('date').str.to_date(strict=False).alias('date_parsed'),
    pl.col('question').str.len_chars().alias('panjang_question'),
).select([
    pl.col('qid').n_unique().alias('qid_unik'),
    pl.col('question').n_unique().alias('question_unik'),
    (pl.len() - pl.col('qid').n_unique()).alias('duplikat_qid'),
    (pl.len() - pl.col('question').n_unique()).alias('duplikat_question'),
    pl.col('date_parsed').min().alias('tanggal_awal'),
    pl.col('date_parsed').max().alias('tanggal_akhir'),
    pl.col('date_parsed').null_count().alias('tanggal_tidak_valid'),
    pl.col('panjang_question').min().alias('panjang_question_min'),
    pl.col('panjang_question').mean().round(2).alias('panjang_question_rata_rata'),
    pl.col('panjang_question').median().alias('panjang_question_median'),
    pl.col('panjang_question').max().alias('panjang_question_max'),
]).collect()
quality_profile

qid_unik,question_unik,duplikat_qid,duplikat_question,tanggal_awal,tanggal_akhir,tanggal_tidak_valid,panjang_question_min,panjang_question_rata_rata,panjang_question_median,panjang_question_max
u32,u32,u32,u32,date,date,u32,u32,f64,f64,u32
232658,311636,642556,563578,2008-08-01,2023-03-04,0,12,748.02,537.0,33832


## Profil DuckDB

DuckDB membaca shard valid langsung dari disk dan menghasilkan ringkasan per kolom.

In [6]:
connection = duckdb.connect()
try:
    profil_duckdb = connection.execute(
        'SUMMARIZE SELECT * FROM read_parquet(?)',
        [[str(file) for file in valid_files]],
    ).to_arrow_table()
    profil_duckdb = pl.from_arrow(profil_duckdb)
finally:
    connection.close()

profil_duckdb

C:\Users\Workshop\AppData\Local\Temp\ipykernel_15816\426995326.py:6: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  ).to_arrow_table()


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""qid""","""BIGINT""","""1""","""74661602""",229684,"""2798205.048132228""","""9044724.94382576""","""26553""","""105124""","""454491""",875214,0.00
"""question""","""VARCHAR""","""!!! I found an interesting sol…","""﻿ What would a political syste…",325828,null,null,null,null,null,875214,0.00
"""date""","""VARCHAR""","""2008/08/01""","""2023/03/04""",5235,null,null,null,null,null,875214,0.00
"""metadata""","""VARCHAR[]""","""['https://3dprinting.meta.stac…","""['https://writers.stackexchang…",322036,null,null,null,null,null,875214,0.00
"""response_j""","""VARCHAR""","""""","""ﷲ = الله ====""",526545,null,null,null,null,null,875214,0.00
"""response_k""","""VARCHAR""","""! Just as well that I have a…","""Watch only produces notificat…",589192,null,null,null,null,null,875214,0.00


## Catatan Milestone 1

Hasil pada salinan lokal (22 Oktober 2026):

- Ada 6 shard berukuran total **960,0 MB**, tetapi `train-00000-of-00006-042e4a2154d9270b.parquet` rusak. Profil saat ini hanya mencakup 5 shard valid: **875.214 baris** dan sekitar **762,8 MB**.
- Kolomnya adalah `qid` (Int64), `question`, `date`, `response_j`, `response_k` (String), serta `metadata` (List(String)).
- Semua kolom pada shard valid memiliki **0 null**; tidak ada kolom dengan null terbanyak.
- Semua `date` valid, dengan rentang **2008-08-01 hingga 2023-03-04**.
- Ada **642.556** `qid` dan **563.578** teks `question` yang berulang; Milestone 2 perlu menentukan aturan deduplikasi berdasarkan kombinasi kolom yang tepat.
- Panjang `question` berkisar 12–33.832 karakter (rata-rata 748,02), sehingga teks ekstrem perlu diperiksa.
- Prioritas: **unduh ulang shard rusak**, lalu jalankan ulang notebook agar statistik mencakup 1.050.257 baris dataset lengkap.